# 08 - WEO Excel SciPy Statistics Lab

This lab uses the IMF World Economic Outlook Excel workbook from Module 4. You will create clean analysis DataFrames from multiple sheets, then run descriptive statistics, confidence intervals, correlations, outlier checks, and hypothesis tests with SciPy.

## Important: Use a Python Notebook Runtime

Run this notebook with a **Python 3** kernel in Jupyter, VS Code, JupyterLab, or Google Colab. Do not run these cells in SQL Server query mode.

This notebook uses the WEO Excel workbook from Module 4. Locally, it looks for `Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx`. In Google Colab, it also checks `/content/WEOApr2026all.xlsx` and can prompt you to upload the file if needed.

## 0. Install Packages

This cell installs the packages needed for this notebook. `%pip` is a notebook command, not SQL syntax. If a package was just updated, Jupyter or Colab may ask you to restart the kernel/runtime before continuing.

In [ ]:
%pip install pandas numpy scipy matplotlib seaborn scikit-learn openpyxl joblib -q

## 1. Imports

Imports load Python libraries into memory. Keeping imports in their own cell makes it easier for beginners to see what tools the notebook uses.

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

## 2. Display and Output Settings

`pd.set_option("display.max_columns", 60)` tells pandas to show up to 60 columns when displaying a table.

`pd.set_option("display.width", 160)` gives pandas more horizontal space before wrapping printed output.

These settings only affect how tables look in the notebook. They do not change the dataset.

In [ ]:
# These settings only control how pandas tables are displayed in the notebook.
# They do not change the data.
pd.set_option("display.max_columns", 60)  # Show up to 60 columns before pandas hides columns with "...".
pd.set_option("display.width", 160)       # Make printed tables wider so rows wrap less often.

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid") if "sns" in globals() else None

## 2.1 Matplotlib Chart Style

The SciPy calculations give numeric evidence. Matplotlib helps us see the shape behind those numbers, such as spread, outliers, and group differences.

In [ ]:
# Use a simple built-in Matplotlib style so charts are readable without extra packages.
plt.style.use("seaborn-v0_8-whitegrid")

## 3. Locate the WEO Workbook

This function checks several possible file paths. The `/content/...` path is used by Google Colab when a file is uploaded to the session.

In [ ]:
def find_weo_workbook():
    """Find the WEO workbook locally, in Colab, or through a Colab upload prompt."""
    candidate_paths = [
        Path("../../../Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("data/WEOApr2026all.xlsx"),
        Path("/content/WEOApr2026all.xlsx"),
        Path("WEOApr2026all.xlsx"),
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    # This block only runs in Google Colab. It lets a learner upload the Excel file manually.
    try:
        from google.colab import files
        print("Upload WEOApr2026all.xlsx from Module 4 to continue.")
        uploaded = files.upload()
        if "WEOApr2026all.xlsx" in uploaded:
            return Path("WEOApr2026all.xlsx")
    except Exception:
        pass

    raise FileNotFoundError(
        "WEOApr2026all.xlsx was not found. Run locally from the repo, or upload the Excel file in Google Colab."
    )

## 4. Open the Workbook and List Sheets

`pd.ExcelFile()` opens the Excel workbook and lets us inspect available sheet names before loading the data.

In [ ]:
WEO_PATH = find_weo_workbook()
print("Using workbook:", WEO_PATH)

excel_file = pd.ExcelFile(WEO_PATH)
print("Workbook sheets:", excel_file.sheet_names)

## 5. Select Relevant Indicators and Standard Column Names

The WEO workbook has many indicators. For this lab, we focus on a smaller set that is useful for macroeconomic analysis. We rename them into consistent `snake_case` columns such as `gdp_growth_pct` and `inflation_pct`.

In [ ]:
# We use short, consistent snake_case names in our analysis DataFrames.
# The original WEO indicator IDs are kept only while extracting the data.
INDICATOR_LABELS = {
    "NGDP_RPCH": "gdp_growth_pct",
    "PCPIPCH": "inflation_pct",
    "LUR": "unemployment_rate",
    "BCA_NGDPD": "current_account_pct_gdp",
    "GGXWDG_NGDP": "government_debt_pct_gdp",
    "NGDPDPC": "gdp_per_capita_usd",
    "NID_NGDP": "investment_pct_gdp",
    "NGSD_NGDP": "savings_pct_gdp",
    "TX_RPCH": "export_volume_growth_pct",
    "TM_RPCH": "import_volume_growth_pct",
}

## 6. Helper Functions for Cleaning and Reshaping

The raw workbook is wide: years are separate columns. The helper functions below:

- standardize column names,
- load the useful sheets,
- convert wide year columns into tidy long format,
- create one `country_macro` DataFrame with one row per country and year.

In [ ]:
def standardize_column_names(frame):
    """Convert non-year column names to snake_case for consistent analysis code."""
    renamed_columns = {}

    for column in frame.columns:
        # WEO year columns arrive as integers such as 1980, 2026, and 2031.
        # Keep them as integers because they are easier to identify and melt later.
        if isinstance(column, int):
            renamed_columns[column] = column
            continue

        clean_name = (
            str(column)
            .strip()
            .lower()
            .replace(".", "_")
            .replace(" ", "_")
            .replace("-", "_")
        )
        renamed_columns[column] = clean_name

    return frame.rename(columns=renamed_columns)


def get_year_columns(frame):
    """Return columns that represent years, for example 1980, 2024, or 2031."""
    return [column for column in frame.columns if isinstance(column, int) or str(column).isdigit()]


def load_weo_sheets(path):
    """Load and standardize the useful WEO workbook sheets."""
    countries = standardize_column_names(pd.read_excel(path, sheet_name="Countries"))
    country_groups = standardize_column_names(pd.read_excel(path, sheet_name="Country Groups"))
    commodity_prices = standardize_column_names(pd.read_excel(path, sheet_name="Commodity Prices"))
    group_composition = standardize_column_names(pd.read_excel(path, sheet_name="Country Group Composition"))

    # This sheet uses compact column names like groupname and countrycode.
    # Rename them once so the rest of the notebook can use readable snake_case names.
    group_composition = group_composition.rename(columns={
        "groupcode": "group_code",
        "groupname": "group_name",
        "groupcode_previous": "group_code_previous",
        "countrycode": "country_id",
        "countryname": "country_name",
        "countrycode_previous": "country_code_previous",
    })

    return countries, country_groups, commodity_prices, group_composition


def make_long_indicator_data(frame, indicator_ids, source_sheet):
    """Convert WEO wide year columns into a tidy row-per-country-year format."""
    year_columns = get_year_columns(frame)
    id_columns = ["country_id", "country", "indicator_id", "indicator", "unit"]

    filtered = frame.loc[frame["indicator_id"].isin(indicator_ids), id_columns + year_columns].copy()

    long_df = filtered.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name="year",
        value_name="value",
    )
    long_df["year"] = long_df["year"].astype(int)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df["source_sheet"] = source_sheet
    return long_df


def make_country_macro_dataframe(countries, group_composition):
    """Create one country-year row with selected WEO indicators as columns."""
    country_long = make_long_indicator_data(countries, list(INDICATOR_LABELS), "Countries")

    country_macro = (
        country_long.pivot_table(
            index=["country_id", "country", "year"],
            columns="indicator_id",
            values="value",
            aggfunc="first",
        )
        .reset_index()
        .rename(columns=INDICATOR_LABELS)
    )

    # Remove the pivot column-axis name so displays do not show an old label above the header row.
    country_macro.columns.name = None

    advanced_codes = set(group_composition.loc[group_composition["group_name"].eq("Advanced Economies"), "country_id"])
    emerging_codes = set(group_composition.loc[group_composition["group_name"].eq("Emerging Market and Developing Economies"), "country_id"])
    ssa_codes = set(group_composition.loc[group_composition["group_name"].eq("Sub-Saharan Africa (SSA)"), "country_id"])

    country_macro["economic_group"] = np.select(
        [country_macro["country_id"].isin(advanced_codes), country_macro["country_id"].isin(emerging_codes)],
        ["Advanced Economies", "Emerging Market and Developing Economies"],
        default="Other",
    )
    country_macro["is_sub_saharan_africa"] = country_macro["country_id"].isin(ssa_codes)

    return country_macro, country_long

## 7. Load Sheets and Build the Analysis DataFrames

The output should show the raw row count from the `Countries` sheet and the number of country-year rows created for analysis. With the current workbook, the analysis table has about 9,398 country-year rows.

In [ ]:
countries_raw, country_groups_raw, commodity_prices_raw, group_composition = load_weo_sheets(WEO_PATH)
country_macro, country_long = make_country_macro_dataframe(countries_raw, group_composition)

print("Countries sheet rows:", len(countries_raw))
print("Country-year analytical rows:", len(country_macro))
display(country_macro.head())

## 8. Recent-Year EDA Summary

EDA means exploratory data analysis. Here we filter to `analysis_year = 2026` and inspect the key indicators before running statistical tests.

In [ ]:
analysis_year = 2026
latest = country_macro.loc[country_macro["year"].eq(analysis_year)].copy()

selected_columns = [
    "country",
    "economic_group",
    "is_sub_saharan_africa",
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_rate",
    "government_debt_pct_gdp",
    "current_account_pct_gdp",
    "gdp_per_capita_usd",
]

# Show a few rows so learners can see the shape of the final DataFrame.
display(latest[selected_columns].head(10))

# describe().T gives one row per numeric column, which is easier to read.
display(latest[selected_columns[3:]].describe().T.round(2))

## 9. SciPy Descriptive Statistics

`stats.describe()` summarizes one numeric variable.

For skewness:

- close to `0` means roughly balanced,
- positive means a longer right tail,
- negative means a longer left tail.

For kurtosis in SciPy, `0` is close to a normal bell-shaped distribution. Large positive values mean more extreme tails/outliers than a normal distribution.

In [ ]:
growth = latest["gdp_growth_pct"].dropna().to_numpy()
growth_stats = stats.describe(growth)

print("Number of countries:", growth_stats.nobs)
print("Minimum GDP growth:", round(growth_stats.minmax[0], 2))
print("Maximum GDP growth:", round(growth_stats.minmax[1], 2))
print("Mean GDP growth:", round(growth_stats.mean, 2))
print("Variance:", round(growth_stats.variance, 2))
print("Skewness:", round(growth_stats.skewness, 2))
print("Kurtosis:", round(growth_stats.kurtosis, 2))

### Interpreting the Descriptive Statistics

With the current workbook, 191 countries have a 2026 GDP growth value. The mean is about **2.96%**, with values ranging from about **-8.65%** to **16.17%**.

The skewness is about **-0.18**, which is close to zero, so the distribution is not strongly tilted left or right. The kurtosis is about **7.28**, which is high. That means the distribution has more extreme values than a normal distribution, so outlier review is important.

## 9.1 Visualize GDP Growth Distribution

A histogram shows the shape of the same `gdp_growth_pct` values summarized by `stats.describe()`. The red dashed line marks the mean.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(growth, bins=20, edgecolor="black", alpha=0.8)
plt.axvline(mean_growth if "mean_growth" in globals() else growth.mean(), color="red", linestyle="--", linewidth=2, label="Mean")
plt.title(f"Distribution of Country GDP Growth, {analysis_year}")
plt.xlabel("GDP Growth (%)")
plt.ylabel("Number of Countries")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_scipy_growth_distribution.png", dpi=150)
plt.show()

**How to read it:** a wide histogram means countries have very different growth rates. Long tails and isolated bars help explain why kurtosis was high in the descriptive statistics.

## 10. Confidence Interval

A confidence interval estimates a likely range for the true average. A 95% confidence interval means that if we repeated this sampling process many times, about 95% of intervals would contain the true mean.

The interval is in the same unit as the variable: percentage points of GDP growth.

In [ ]:
mean_growth = np.mean(growth)
standard_error = stats.sem(growth)
confidence_interval = stats.t.interval(
    confidence=0.95,
    df=len(growth) - 1,
    loc=mean_growth,
    scale=standard_error,
)

print(f"{analysis_year} mean GDP growth:", round(mean_growth, 2))
print("95% confidence interval:", tuple(round(float(value), 2) for value in confidence_interval))

### Interpreting the Confidence Interval

The current workbook gives a 95% interval of about **2.61% to 3.32%** for the average 2026 country-level GDP growth rate. This means the estimated average is around **2.96%**, but normal sampling uncertainty puts the likely average in that range.

## 10.1 Visualize the Confidence Interval

This chart shows the sample mean GDP growth and its 95% confidence interval. The dot is the mean. The horizontal line is the interval.

In [ ]:
plt.figure(figsize=(7, 2.5))
plt.errorbar(
    x=mean_growth,
    y=0,
    xerr=[[mean_growth - confidence_interval[0]], [confidence_interval[1] - mean_growth]],
    fmt="o",
    color="darkblue",
    ecolor="steelblue",
    elinewidth=3,
    capsize=8,
)
plt.yticks([])
plt.title(f"Mean GDP Growth with 95% Confidence Interval, {analysis_year}")
plt.xlabel("GDP Growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_scipy_growth_confidence_interval.png", dpi=150)
plt.show()

**How to read it:** if the interval is narrow, the average is estimated more precisely. Here, the interval is around 2.61% to 3.32%.

## 11. Correlation: Inflation and GDP Growth

Correlation ranges from `-1` to `1`:

- `1` means a perfect positive relationship,
- `0` means no linear relationship,
- `-1` means a perfect negative relationship.

The p-value ranges from `0` to `1`. A p-value below `0.05` is commonly treated as statistically significant evidence against "no relationship".

In [ ]:
corr_df = latest[["gdp_growth_pct", "inflation_pct"]].dropna()

# Pearson checks a linear relationship.
pearson_corr, pearson_p = stats.pearsonr(corr_df["inflation_pct"], corr_df["gdp_growth_pct"])

# Spearman checks whether the ranking of one variable moves with the ranking of the other.
spearman_corr, spearman_p = stats.spearmanr(corr_df["inflation_pct"], corr_df["gdp_growth_pct"])

print("Pearson correlation:", round(pearson_corr, 3), "p-value:", round(pearson_p, 4))
print("Spearman correlation:", round(spearman_corr, 3), "p-value:", round(spearman_p, 4))

### Interpreting the Correlation Output

The current results show Pearson correlation around **-0.025** with p-value around **0.736**, and Spearman correlation around **0.115** with p-value around **0.114**.

Both p-values are above `0.05`, so this notebook does **not** find statistically significant evidence of a simple inflation-growth relationship in this 2026 cross-section. This does not prove no economic relationship exists; it only means this simple two-variable check is weak for this year.

## 11.1 Visualize the Correlation Check

This scatter plot uses the same data as the Pearson and Spearman correlation tests. Each point is one country.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(corr_df["inflation_pct"], corr_df["gdp_growth_pct"], alpha=0.7)
plt.axhline(0, color="black", linewidth=0.8)
plt.title(f"Inflation vs GDP Growth, {analysis_year}")
plt.xlabel("Inflation (%)")
plt.ylabel("GDP Growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_scipy_inflation_growth_scatter.png", dpi=150)
plt.show()

**How to read it:** a strong relationship would show a clear upward or downward pattern. The current test results showed weak correlations and high p-values, so the scatter should not be interpreted as a strong relationship.

## 12. Outlier Review with Z-Scores

A z-score tells how many standard deviations a value is from the mean.

A common beginner rule is:

- `abs(z_score) <= 2`: usually not extreme,
- `abs(z_score) > 2`: review as a possible outlier,
- `abs(z_score) > 3`: very unusual compared with the rest of the data.

In [ ]:
latest["growth_z_score"] = stats.zscore(latest["gdp_growth_pct"], nan_policy="omit")
latest["growth_outlier_flag"] = latest["growth_z_score"].abs() > 2

outlier_columns = ["country", "economic_group", "gdp_growth_pct", "growth_z_score"]
outliers = latest.loc[latest["growth_outlier_flag"], outlier_columns].sort_values("growth_z_score")
display(outliers)

### Interpreting the Outliers

The current workbook flags countries such as **Qatar**, **Iraq**, **Iran**, and **Guyana** as growth outliers for 2026. These rows should not be deleted automatically. Outliers may reflect real economic events, data revisions, or forecast assumptions. The correct next step is to review them in context.

## 12.1 Visualize Growth Outliers

This bar chart shows the countries flagged by the z-score outlier rule. Bars far from zero deserve review.

In [ ]:
plt.figure(figsize=(9, 5))
colors = ["crimson" if value < 0 else "seagreen" for value in outliers["growth_z_score"]]
plt.barh(outliers["country"], outliers["growth_z_score"], color=colors)
plt.axvline(-2, color="black", linestyle="--", linewidth=1, label="-2 z-score")
plt.axvline(2, color="black", linestyle="--", linewidth=1, label="+2 z-score")
plt.title(f"GDP Growth Outliers by Z-Score, {analysis_year}")
plt.xlabel("GDP Growth Z-Score")
plt.ylabel("Country")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_scipy_growth_outlier_zscores.png", dpi=150)
plt.show()

**How to read it:** negative z-scores are unusually low growth values; positive z-scores are unusually high growth values. This chart makes the flagged outliers easier to inspect than a table alone.

## 13. Hypothesis Test: Economic Group Difference

This t-test compares average 2026 GDP growth between advanced economies and emerging/developing economies.

The p-value is between `0` and `1`. Very small values may appear in scientific notation. For example, `5.395e-07` means `0.0000005395`, not `5.395`.

In [ ]:
advanced_growth = latest.loc[latest["economic_group"].eq("Advanced Economies"), "gdp_growth_pct"].dropna()
emerging_growth = latest.loc[latest["economic_group"].eq("Emerging Market and Developing Economies"), "gdp_growth_pct"].dropna()

t_stat, p_value = stats.ttest_ind(advanced_growth, emerging_growth, equal_var=False)

print("Advanced economies mean:", round(advanced_growth.mean(), 2))
print("Emerging/developing economies mean:", round(emerging_growth.mean(), 2))
print("t-statistic:", round(t_stat, 3))
print("p-value:", p_value)

if p_value < 0.05:
    print("Interpretation: the group difference is statistically significant at the 5% level.")
else:
    print("Interpretation: the group difference is not statistically significant at the 5% level.")

### Interpreting the T-Test

The current output shows advanced economies averaging about **1.85%** growth and emerging/developing economies averaging about **3.29%** growth. The p-value is about **5.40e-07**, which is far below `0.05`.

That means the observed group difference is statistically significant in this workbook. In plain language: the two groups show meaningfully different average projected GDP growth for 2026.

## 13.1 Visualize the Group Difference

The t-test compares two group means. This box plot shows the same comparison visually.

In [ ]:
group_plot_df = latest[latest["economic_group"].isin(["Advanced Economies", "Emerging Market and Developing Economies"])].copy()

advanced_values = group_plot_df.loc[group_plot_df["economic_group"].eq("Advanced Economies"), "gdp_growth_pct"].dropna()
emerging_values = group_plot_df.loc[group_plot_df["economic_group"].eq("Emerging Market and Developing Economies"), "gdp_growth_pct"].dropna()

plt.figure(figsize=(9, 4))
plt.boxplot(
    [advanced_values, emerging_values],
    labels=["Advanced Economies", "Emerging/Developing"],
    patch_artist=True,
    boxprops={"facecolor": "lightsteelblue"},
    medianprops={"color": "darkred", "linewidth": 2},
)
plt.title(f"GDP Growth Distribution by Economic Group, {analysis_year}")
plt.ylabel("GDP Growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_scipy_group_growth_boxplot.png", dpi=150)
plt.show()

**How to read it:** the median line and box position show that the emerging/developing group generally has higher projected growth in this workbook. The t-test confirms that the mean difference is statistically significant.

## 14. Export Statistical Outputs

Exports let learners use notebook results in reports or later labs.

In [ ]:
summary_path = OUTPUT_DIR / "weo_scipy_2026_summary.csv"
outliers_path = OUTPUT_DIR / "weo_scipy_growth_outliers.csv"

latest[selected_columns[3:]].describe().T.round(2).to_csv(summary_path)
outliers.to_csv(outliers_path, index=False)

print("Saved:", summary_path)
print("Saved:", outliers_path)

print("The chart files were saved in:", OUTPUT_DIR)